# Long Document Summarizer | Map-Reduce Agents

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List
from concurrent.futures import ThreadPoolExecutor
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

class MapReduceState(TypedDict):
    document: str
    chunks: List[str]
    chunk_summaries: List[str]
    final_summary: str

In [4]:
def split_document(state: MapReduceState) -> dict:
    """Split the document into overlapping chunks."""
    text = state["document"]
    chunk_size = 2000  # characters per chunk
    overlap = 200
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start = end - overlap
        if start + overlap >= len(text):
            break
    return {"chunks": chunks}

In [5]:
def map_summarize(state: MapReduceState) -> dict:
    """Map phase: summarize each chunk in parallel."""
    def summarize_chunk(chunk_with_index: tuple) -> str:
        idx, chunk = chunk_with_index
        response = model.invoke(
            f"Summarize this section (part {idx + 1}) in 2-3 key points. "
            f"Focus on the most important information:\n\n{chunk}"
        )
        return response.content

    indexed_chunks = list(enumerate(state["chunks"]))
    with ThreadPoolExecutor(max_workers=5) as executor:
        summaries = list(executor.map(summarize_chunk, indexed_chunks))
    return {"chunk_summaries": summaries}

In [6]:
def reduce_combine(state: MapReduceState) -> dict:
    """Reduce phase: combine chunk summaries into a final summary."""
    all_summaries = "\n\n---\n\n".join(
        f"Section {i+1}:\n{s}" for i, s in enumerate(state["chunk_summaries"])
    )
    response = model.invoke(
        f"Synthesize these section summaries into a cohesive final summary. "
        f"Identify key themes across sections, resolve any contradictions, and "
        f"highlight the most critical findings. Do NOT simply concatenate — "
        f"produce an integrated narrative with logical flow.\n\n"
        f"{all_summaries}"
    )
    return {"final_summary": response.content}

In [7]:
# Build graph
graph = StateGraph(MapReduceState)
graph.add_sequence([("split", split_document), ("map", map_summarize), ("reduce", reduce_combine)])
graph.add_edge(START, "split")
graph.add_edge("reduce", END)

map_reduce = graph.compile()

In [8]:
plot_mermaid(map_reduce)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	split(split)
	map(map)
	reduce(reduce)
	__end__([<p>__end__</p>]):::last
	__start__ --> split;
	map --> reduce;
	split --> map;
	reduce --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [9]:
# Example: summarize a long text (repeated to simulate a long document)
long_document = (
    "Artificial Intelligence has evolved dramatically over the past decade. "
    "Starting with narrow AI systems that excelled at specific tasks like image recognition "
    "and language translation, the field has progressed toward more general capabilities. "
    "Large Language Models represent a paradigm shift, demonstrating emergent abilities "
    "in reasoning, coding, and multi-step problem solving. "
    "The emergence of agentic AI systems -- where LLMs autonomously use tools, plan, and "
    "execute complex workflows -- marks the next frontier. These systems combine the "
    "language understanding of LLMs with the ability to take actions in the real world. "
) * 20  # Simulate a long document

result = map_reduce.invoke({"document": long_document})
print(result["final_summary"])

Over the past decade, the field of Artificial Intelligence (AI) has seen a transformative evolution from narrow, task-specific systems to platforms with increasingly general capabilities. This shift is prominently exemplified by the development and deployment of Large Language Models (LLMs), which represent a significant paradigm shift in AI. LLMs have demonstrated emergent abilities beyond their original design, showcasing advanced skills in reasoning, coding, and multi-step problem solving. These capabilities mark a notable departure from the limitations of specialized systems like image recognition and language translation, indicating a trajectory towards more sophisticated AI with general intelligence.

Complementing this development is the emergence of agentic AI systems, which represent the next significant frontier in AI technology. These systems further extend the functionality of LLMs by combining advanced language understanding with autonomous action-taking capabilities in re

In [10]:
# Streaming

stream_invoke(map_reduce, {"document": long_document})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'document': 'Artificial Intelligence has evolved dramatically over the past decade. Starting with narrow AI systems that excelled at specific tasks like image recognition and language translation, the field has progressed toward more general capabilities. Large Language Models represent a paradigm shift, demonstrating emergent abilities in reasoning, coding, and multi-step problem solving. The emergence of agentic AI systems -- where LLMs autonomously use tools, plan, and execute complex workflows -- marks the next frontier. These systems combine the language understanding of LLMs with the ability to take actions in the real world. Artificial Intelligence has evolved dramatically over the past decade. Starting with narrow AI systems that excelled at specific tasks like image recognition and language translation, the field has progressed toward more general capabilities. Large Language Models represent a paradigm shift, demonstrating emergent abilities in reasoning, coding, and multi-s